# Mask R-CNN Training Pipeline: Crop → SAM → Train

This notebook covers the complete workflow for training a Mask R-CNN model on microplastic crops with SAM-generated segmentation masks.

**Strategy:** Data is copied from Google Drive to Colab local storage for fast, reliable processing. Results are copied back to Drive at the end.

**Pipeline Steps:**
1. **Install Dependencies** — pip install required packages
2. **Import Libraries** — Load all Python modules
3. **Define Functions** — Embedded helper functions (no `src/` needed)
4. **Configure & Copy Data** — Copy YOLO data + SAM checkpoint to local storage
5. **Explore Dataset** — Inspect images, labels, class distribution
6. **Crop Images** — Extract crops from YOLO ground-truth labels
7. **Verify Crops** — Check crop counts and dimensions
8. **Initialize SAM** — Load Segment Anything Model (ViT-H)
9. **Generate SAM Masks** — Create segmentation masks for each crop
10. **Post-Process Masks** — Quality check and filtering
11. **Convert to COCO Format** — Prepare annotations for evaluation
12. **Build Dataset & Model** — CropDataset + Mask R-CNN (ResNet50-FPN)
13. **Train Mask R-CNN** — Full training loop with checkpointing
14. **Evaluate & Visualize** — Metrics, loss curves, prediction overlays
15. **Copy Results to Drive** — Save models back to Google Drive

## Setup: Mount Google Drive

This notebook is designed to run on Google Colab with data stored in Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify mount
import os
if os.path.exists('/content/drive/MyDrive/mp-detect'):
    print("✓ Google Drive mounted successfully")
    print(f"✓ Project directory found: /content/drive/MyDrive/mp-detect")
else:
    print("✗ Project directory not found. Please upload mp-detect folder to MyDrive")

## 1. Install and Import Dependencies

Install required packages and import all libraries needed for dataset processing, SAM inference, and Mask R-CNN training.

In [ ]:
# ============================================================================
# INSTALL DEPENDENCIES (run this cell first!)
# ============================================================================
!pip install -q torch torchvision pycocotools opencv-python-headless
!pip install -q albumentations ultralytics timm tqdm matplotlib
!pip install -q segment-anything
print("✓ All packages installed")

In [ ]:
# ============================================================================
# IMPORT LIBRARIES
# ============================================================================
import json
import cv2
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt
import random

# PyTorch and torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torch.utils.data import Dataset, DataLoader

# Albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Segment Anything (only needed later for SAM step)
from segment_anything import SamAutomaticMaskGenerator, SamPredictor, sam_model_registry

print("✓ All libraries imported")

In [ ]:
# ============================================================================
# EMBEDDED FUNCTIONS (no src/ imports needed)
# ============================================================================

# Configuration constants
NUM_CLASSES = 4
CLASS_NAMES = ['background', 'fiber', 'film', 'fragment']
YOLO_TO_MASKRCNN = {0: 1, 1: 2, 2: 3}

# CropDataset class
class CropDataset(Dataset):
    """Dataset of cropped detections for Mask R-CNN training."""
    
    CLASS_NAME_TO_YOLO_ID = {'fiber': 0, 'film': 1, 'fragment': 2}
    
    def __init__(self, crops_dir: str, transforms=None):
        self.crops_dir = Path(crops_dir)
        self.transforms = transforms
        
        ann_file = self.crops_dir / 'annotations.json'
        has_flat_images = (self.crops_dir / 'images').is_dir()
        has_class_dirs = any((self.crops_dir / c).is_dir() for c in self.CLASS_NAME_TO_YOLO_ID)
        
        if ann_file.exists():
            with open(ann_file) as f:
                self.annotations = json.load(f)
            if has_flat_images:
                self.images_dir = self.crops_dir / 'images'
            elif has_class_dirs:
                self.images_dir = None
            else:
                self.images_dir = self.crops_dir / 'images'
        elif has_class_dirs:
            self.annotations = {}
            self.images_dir = None
            for cls_name, cls_id in self.CLASS_NAME_TO_YOLO_ID.items():
                cls_dir = self.crops_dir / cls_name
                if not cls_dir.is_dir():
                    continue
                for img_file in sorted(cls_dir.glob('*.png')):
                    crop = cv2.imread(str(img_file))
                    if crop is None:
                        continue
                    h, w = crop.shape[:2]
                    self.annotations[img_file.name] = {
                        'source_image': '',
                        'class_id': cls_id,
                        'class_name': cls_name,
                        'yolo_confidence': 1.0,
                        'rel_box': [0, 0, w, h],
                        'crop_size': [w, h]
                    }
            print(f"Auto-generated annotations for {len(self.annotations)} crops")
        else:
            raise FileNotFoundError(f"No annotations.json or class subdirs found in {crops_dir}")
        
        self.samples = list(self.annotations.keys())
        print(f"Loaded {len(self.samples)} crop samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample_name = self.samples[idx]
        ann = self.annotations[sample_name]
        
        if self.images_dir is not None:
            img_path = self.images_dir / sample_name
        else:
            cls_name = ann.get('class_name', '')
            img_path = self.crops_dir / cls_name / sample_name
        
        image = cv2.imread(str(img_path))
        if image is None:
            raise FileNotFoundError(f"Could not load: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        h, w = image.shape[:2]
        
        # Load mask — prefer SAM mask, fall back to ellipse
        mask = None
        mask_file = ann.get('mask_file')
        masks_dir = self.crops_dir / 'masks'
        
        if mask_file and (masks_dir / mask_file).exists():
            raw = cv2.imread(str(masks_dir / mask_file), cv2.IMREAD_GRAYSCALE)
            if raw is not None:
                mask = (raw > 127).astype(np.uint8)
        
        if mask is None:
            default_mask = masks_dir / sample_name.replace('.png', '_mask.png')
            if default_mask.exists():
                raw = cv2.imread(str(default_mask), cv2.IMREAD_GRAYSCALE)
                if raw is not None:
                    mask = (raw > 127).astype(np.uint8)
        
        if mask is None:
            mask = self._create_ellipse_mask(h, w, ann.get('rel_box'))
        
        if mask.shape[:2] != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
        
        ys, xs = np.where(mask > 0)
        if len(xs) > 0 and len(ys) > 0:
            box = [xs.min(), ys.min(), xs.max(), ys.max()]
        else:
            margin = min(h, w) // 10
            box = [margin, margin, w - margin, h - margin]
        
        class_id = YOLO_TO_MASKRCNN[ann['class_id']]
        
        boxes = np.array([box], dtype=np.float32)
        labels = np.array([class_id], dtype=np.int64)
        masks = np.array([mask], dtype=np.uint8)
        
        if self.transforms:
            transformed = self.transforms(
                image=image,
                bboxes=boxes.tolist(),
                masks=list(masks),
                class_labels=labels.tolist()
            )
            image = transformed['image']
            if len(transformed['bboxes']) > 0:
                boxes = np.array(transformed['bboxes'], dtype=np.float32)
                labels = np.array(transformed['class_labels'], dtype=np.int64)
                masks = np.array(transformed['masks'], dtype=np.uint8)
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
        
        target = {
            'boxes': torch.as_tensor(boxes, dtype=torch.float32),
            'labels': torch.as_tensor(labels, dtype=torch.int64),
            'masks': torch.as_tensor(masks, dtype=torch.uint8),
            'image_id': torch.tensor([idx]),
            'area': torch.as_tensor([(box[2]-box[0])*(box[3]-box[1]) for box in boxes], dtype=torch.float32),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64)
        }
        
        return image, target
    
    def _create_ellipse_mask(self, h: int, w: int, rel_box=None):
        mask = np.zeros((h, w), dtype=np.uint8)
        if rel_box:
            x1, y1, x2, y2 = rel_box
            center = ((x1 + x2) // 2, (y1 + y2) // 2)
            axes = ((x2 - x1) // 2, (y2 - y1) // 2)
        else:
            center = (w // 2, h // 2)
            axes = (int(w * 0.4), int(h * 0.4))
        if axes[0] > 0 and axes[1] > 0:
            cv2.ellipse(mask, center, axes, 0, 0, 360, 1, -1)
        return mask


def get_transforms(train=True, img_size=128):
    """Get augmentation transforms for crops."""
    if train:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'], min_visibility=0.3))
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'], min_visibility=0.3))


def collate_fn(batch):
    """Custom collate for Mask R-CNN."""
    return tuple(zip(*batch))


def get_model(num_classes: int, pretrained: bool = True):
    """Create Mask R-CNN model."""
    if pretrained:
        model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    else:
        model = maskrcnn_resnet50_fpn(weights=None)
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    
    return model


def prepare_from_yolo_labels(yolo_dir: str = 'data/yolo', output_dir: str = 'data/crops',
                              padding: int = 20, splits=('train', 'val')):
    """Generate crops from YOLO ground-truth labels."""
    yolo_path = Path(yolo_dir)
    out_path = Path(output_dir)
    (out_path / 'images').mkdir(parents=True, exist_ok=True)
    for cls_name in CLASS_NAMES[1:]:
        (out_path / cls_name).mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print("PREPARING CROPS FROM YOLO GROUND-TRUTH LABELS")
    print(f"{'='*60}")
    print(f"YOLO dir:  {yolo_dir}")
    print(f"Output:    {output_dir}")
    print(f"Padding:   {padding}px")
    print(f"Splits:    {splits}")
    print(f"{'='*60}\n")

    annotations = {}
    crop_count = 0
    class_counts = {0: 0, 1: 0, 2: 0}

    for split in splits:
        img_dir = yolo_path / 'images' / split
        lbl_dir = yolo_path / 'labels' / split

        if not img_dir.exists():
            print(f"Skipping '{split}' — {img_dir} not found")
            continue

        image_files = sorted(list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg')))
        print(f"[{split}] {len(image_files)} images")

        for img_path in tqdm(image_files, desc=f"Processing {split}"):
            image = cv2.imread(str(img_path))
            if image is None:
                continue

            h, w = image.shape[:2]
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue

            with open(lbl_path) as f:
                lines = [l.strip() for l in f if l.strip()]

            for line in lines:
                parts = line.split()
                if len(parts) < 5:
                    continue

                cls_id = int(float(parts[0]))
                cx_n, cy_n, bw_n, bh_n = map(float, parts[1:5])

                cx, cy, bw, bh = cx_n*w, cy_n*h, bw_n*w, bh_n*h
                det_x1, det_y1 = int(cx-bw/2), int(cy-bh/2)
                det_x2, det_y2 = int(cx+bw/2), int(cy+bh/2)

                x1 = max(0, det_x1 - padding)
                y1 = max(0, det_y1 - padding)
                x2 = min(w, det_x2 + padding)
                y2 = min(h, det_y2 + padding)

                crop_img = image[y1:y2, x1:x2]
                if crop_img.size == 0 or crop_img.shape[0] < 10 or crop_img.shape[1] < 10:
                    continue

                cls_name = CLASS_NAMES[YOLO_TO_MASKRCNN[cls_id]]
                crop_filename = f"{img_path.stem}_gt{crop_count:04d}_{cls_name}.png"

                cv2.imwrite(str(out_path / 'images' / crop_filename), crop_img)
                cv2.imwrite(str(out_path / cls_name / crop_filename), crop_img)

                rel_x1, rel_y1 = det_x1 - x1, det_y1 - y1
                rel_x2, rel_y2 = det_x2 - x1, det_y2 - y1

                annotations[crop_filename] = {
                    'source_image': img_path.name,
                    'split': split,
                    'class_id': cls_id,
                    'class_name': cls_name,
                    'yolo_confidence': 1.0,
                    'rel_box': [rel_x1, rel_y1, rel_x2, rel_y2],
                    'crop_size': [crop_img.shape[1], crop_img.shape[0]]
                }

                class_counts[cls_id] += 1
                crop_count += 1

    with open(out_path / 'annotations.json', 'w') as f:
        json.dump(annotations, f, indent=2)

    print(f"\n{'='*60}")
    print("CROP PREPARATION COMPLETE")
    print(f"{'='*60}")
    print(f"Total:    {crop_count}")
    print(f"  Fiber:    {class_counts[0]}")
    print(f"  Film:     {class_counts[1]}")
    print(f"  Fragment: {class_counts[2]}")
    print(f"Saved to: {out_path}")
    print(f"{'='*60}\n")


def convert_to_coco_format(crops_dir: str, output_file: str = None):
    """Convert SAM annotations to COCO format."""
    try:
        from pycocotools import mask as mask_utils
    except ImportError:
        print("Warning: pycocotools not installed. Skipping COCO conversion.")
        return None
    
    crops_path = Path(crops_dir)
    
    with open(crops_path / 'annotations.json') as f:
        annotations = json.load(f)
    
    coco = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": 1, "name": "fiber"},
            {"id": 2, "name": "film"},
            {"id": 3, "name": "fragment"}
        ]
    }
    
    ann_id = 1
    for img_id, (sample_name, ann) in enumerate(annotations.items(), start=1):
        img_path = crops_path / 'images' / sample_name
        if not img_path.exists():
            continue
        
        image = cv2.imread(str(img_path))
        h, w = image.shape[:2]
        
        coco["images"].append({
            "id": img_id,
            "file_name": sample_name,
            "width": w,
            "height": h
        })
        
        mask_file = ann.get('mask_file')
        if mask_file:
            mask_path = crops_path / 'masks' / mask_file
            if mask_path.exists():
                mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                mask_binary = (mask > 127).astype(np.uint8)
                
                rle = mask_utils.encode(np.asfortranarray(mask_binary))
                rle['counts'] = rle['counts'].decode('utf-8')
                
                contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if contours:
                    x, y, bw, bh = cv2.boundingRect(contours[0])
                    area = cv2.contourArea(contours[0])
                else:
                    x, y, bw, bh = 0, 0, w, h
                    area = w * h
                
                class_id = YOLO_TO_MASKRCNN.get(ann.get('class_id', 0), 1)
                
                coco["annotations"].append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": class_id,
                    "segmentation": rle,
                    "area": float(area),
                    "bbox": [x, y, bw, bh],
                    "iscrowd": 0
                })
                ann_id += 1
    
    if output_file is None:
        output_file = str(crops_path / 'coco_annotations.json')
    
    with open(output_file, 'w') as f:
        json.dump(coco, f, indent=2)
    
    print(f"COCO annotations saved to: {output_file}")
    return output_file

print("✓ All functions defined")

In [ ]:
# ============================================================================
# Configuration — Robust Copy with Retry + Auto-Remount
# ============================================================================
import shutil, time

# Google Drive source paths
DRIVE_ROOT = Path('/content/drive/MyDrive/mp-detect')
DRIVE_YOLO = DRIVE_ROOT / 'data/yolo'
DRIVE_SAM_CKPT = DRIVE_ROOT / 'sam_vit_h_4b8939.pth'
DRIVE_EXPERIMENTS = DRIVE_ROOT / 'experiments'

# Local Colab paths (fast SSD, no disconnects)
LOCAL_ROOT = Path('/content/mp_data')
YOLO_DIR = LOCAL_ROOT / 'yolo'
CROPS_DIR = LOCAL_ROOT / 'crops'
SAM_OUTPUT_DIR = LOCAL_ROOT / 'crops_sam'
SAM_CHECKPOINT = LOCAL_ROOT / 'sam_vit_h_4b8939.pth'
SAVE_DIR = LOCAL_ROOT / 'experiments'

# ---------------------------------------------------------------------------
# Resilient copy: file-by-file with retries & auto Drive remount
# ---------------------------------------------------------------------------
def _remount_drive():
    """Force-remount Google Drive after a disconnect."""
    print("  ⟳ Remounting Google Drive...")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        time.sleep(2)
    except Exception:
        pass
    from google.colab import drive as _d
    _d.mount('/content/drive', force_remount=True)
    time.sleep(3)
    print("  ✓ Drive remounted")

def robust_copy_file(src: Path, dst: Path, max_retries=3):
    """Copy a single file with retries; remount Drive on Errno 107."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, max_retries + 1):
        try:
            shutil.copy2(str(src), str(dst))
            return True
        except OSError as e:
            if e.errno == 107 or 'Transport endpoint' in str(e):
                print(f"  ⚠ Drive disconnected copying {src.name} (attempt {attempt}/{max_retries})")
                _remount_drive()
            else:
                raise
    print(f"  ✗ FAILED after {max_retries} retries: {src.name}")
    return False

def robust_copytree(src_dir: Path, dst_dir: Path):
    """Walk src_dir and copy every file with retry logic."""
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    copied, skipped, failed = 0, 0, 0
    all_files = list(src_dir.rglob('*'))
    files_only = [f for f in all_files if f.is_file()]
    print(f"  Found {len(files_only)} files to copy")
    for i, src_file in enumerate(files_only):
        rel = src_file.relative_to(src_dir)
        dst_file = dst_dir / rel
        if dst_file.exists() and dst_file.stat().st_size > 0:
            skipped += 1
            continue
        ok = robust_copy_file(src_file, dst_file)
        if ok:
            copied += 1
        else:
            failed += 1
        if (copied + skipped + failed) % 200 == 0:
            print(f"  ... progress: {copied} copied, {skipped} skipped, {failed} failed / {len(files_only)}")
    print(f"  Done: {copied} copied, {skipped} already existed, {failed} failed")
    return failed == 0

# Copy YOLO dataset from Drive → local (resumable)
yolo_marker = YOLO_DIR / '.copy_complete'
if not yolo_marker.exists():
    YOLO_DIR.mkdir(parents=True, exist_ok=True)
    print("Copying YOLO dataset to local storage (resilient, resumable)...")
    if robust_copytree(DRIVE_YOLO, YOLO_DIR):
        yolo_marker.touch()                       # mark complete
        print(f"  ✓ YOLO copy complete → {YOLO_DIR}")
    else:
        print("  ⚠ Some files failed — re-run this cell to resume")
else:
    print(f"✓ YOLO data already local: {YOLO_DIR}")

# Copy SAM checkpoint: Drive → local, or download from Meta if not on Drive
SAM_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'
if not SAM_CHECKPOINT.exists():
    SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    if DRIVE_SAM_CKPT.exists():
        print("Copying SAM checkpoint from Drive...")
        robust_copy_file(DRIVE_SAM_CKPT, SAM_CHECKPOINT)
        print(f"  ✓ SAM checkpoint copied → {SAM_CHECKPOINT}")
    else:
        print(f"SAM checkpoint not found on Drive, downloading from Meta (~2.4 GB)...")
        import urllib.request
        urllib.request.urlretrieve(SAM_URL, str(SAM_CHECKPOINT))
        print(f"  ✓ SAM checkpoint downloaded → {SAM_CHECKPOINT}")
else:
    print(f"✓ SAM checkpoint already local: {SAM_CHECKPOINT}")

# Create output directories
CROPS_DIR.mkdir(parents=True, exist_ok=True)
SAM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
SAM_MODEL_TYPE = 'vit_h'
CROP_PADDING = 20

# Mask R-CNN training config
MASKRCNN_EPOCHS = 50
MASKRCNN_BATCH_SIZE = 8
MASKRCNN_LR = 0.001
CROP_SIZE = 128
NUM_CLASSES = 4

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {DEVICE}")

print(f"\nLocal Data Paths (fast, no Drive disconnects):")
print(f"  YOLO:       {YOLO_DIR}")
print(f"  Crops:      {CROPS_DIR}")
print(f"  SAM output: {SAM_OUTPUT_DIR}")
print(f"  SAM model:  {SAM_CHECKPOINT}")
print(f"  Experiments: {SAVE_DIR}")

## 2. Load and Explore the Dataset

Inspect the YOLO augmented dataset structure — images, labels, class distribution, and sample images.

In [ ]:
# Explore the YOLO augmented dataset
yolo_path = Path(YOLO_DIR)

# Count images and labels per split
for split in ['train', 'val']:
    img_dir = yolo_path / 'images' / split
    lbl_dir = yolo_path / 'labels' / split
    
    if img_dir.exists():
        imgs = list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg'))
        lbls = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
        print(f"[{split}] Images: {len(imgs)}, Labels: {len(lbls)}")
    else:
        print(f"[{split}] Not found")

# Count annotations per class
class_counts = defaultdict(int)
total_objects = 0

for split in ['train', 'val']:
    lbl_dir = yolo_path / 'labels' / split
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob('*.txt'):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    cls_name = CLASS_NAMES[YOLO_TO_MASKRCNN.get(cls_id, 0)]
                    class_counts[cls_name] += 1
                    total_objects += 1

print(f"\nTotal annotated objects: {total_objects}")
for cls, count in sorted(class_counts.items()):
    print(f"  {cls}: {count} ({count/total_objects*100:.1f}%)")

# Show sample images with bounding boxes
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Sample Images from YOLO Augmented Dataset', fontsize=14)

img_dir = yolo_path / 'images' / 'train'
lbl_dir = yolo_path / 'labels' / 'train'
sample_imgs = random.sample(list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg')), 
                           min(4, len(list(img_dir.iterdir()))))

colors = {'fiber': (255, 0, 0), 'film': (0, 255, 0), 'fragment': (0, 0, 255)}

for ax, img_path in zip(axes, sample_imgs):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Draw bounding boxes from label
    lbl_path = lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    cx, cy, bw, bh = float(parts[1])*w, float(parts[2])*h, float(parts[3])*w, float(parts[4])*h
                    x1, y1 = int(cx - bw/2), int(cy - bh/2)
                    x2, y2 = int(cx + bw/2), int(cy + bh/2)
                    cls_name = CLASS_NAMES[YOLO_TO_MASKRCNN[cls_id]]
                    color = colors.get(cls_name, (255, 255, 0))
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(img, cls_name, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Crop Images from the Dataset

Extract individual microplastic crops from YOLO ground-truth labels. Each detection is cropped with padding and saved along with its class annotation. This uses `prepare_from_yolo_labels()` from the project codebase.

In [ ]:
# Crop from YOLO ground-truth labels
# Each detection bbox is extracted with CROP_PADDING pixels of context
prepare_from_yolo_labels(
    yolo_dir=str(YOLO_DIR),
    output_dir=str(CROPS_DIR),
    padding=CROP_PADDING
)

In [ ]:
# Verify generated crops
crops_path = Path(CROPS_DIR)

with open(crops_path / 'annotations.json') as f:
    crop_annotations = json.load(f)

print(f"Total crops: {len(crop_annotations)}")

# Count per class
crop_class_counts = defaultdict(int)
for ann in crop_annotations.values():
    crop_class_counts[ann['class_name']] += 1

for cls, count in sorted(crop_class_counts.items()):
    print(f"  {cls}: {count}")

# Show sample crops per class
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Sample Crops by Class', fontsize=14)

by_class = defaultdict(list)
for name, ann in crop_annotations.items():
    by_class[ann['class_name']].append(name)

for row, cls_name in enumerate(['fiber', 'film', 'fragment']):
    samples = random.sample(by_class[cls_name], min(4, len(by_class[cls_name])))
    for col, name in enumerate(samples):
        img = cv2.imread(str(crops_path / 'images' / name))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(img)
        axes[row, col].set_title(f"{cls_name}\n{img.shape[1]}x{img.shape[0]}px", fontsize=9)
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()

# Size distribution
sizes = [(ann['crop_size'][0], ann['crop_size'][1]) for ann in crop_annotations.values()]
widths, heights = zip(*sizes)
print(f"\nCrop dimensions — Width: {np.min(widths)}-{np.max(widths)}px (mean {np.mean(widths):.0f})")
print(f"                  Height: {np.min(heights)}-{np.max(heights)}px (mean {np.mean(heights):.0f})")

## 4. Initialize SAM Model for Mask Generation

Load Meta's Segment Anything Model (SAM) with the ViT-H backbone for maximum mask accuracy. SAM generates pixel-precise segmentation masks using point and bounding box prompts derived from our crop annotations.

In [ ]:
# Load SAM model
print(f"Loading SAM model ({SAM_MODEL_TYPE}) from: {SAM_CHECKPOINT}")
print(f"Device: {DEVICE}")

assert SAM_CHECKPOINT.exists(), f"SAM checkpoint not found: {SAM_CHECKPOINT}"

sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=str(SAM_CHECKPOINT))
sam.to(DEVICE)
sam.eval()

# Initialize predictor (point/box prompts — better for our centered crops)
sam_predictor = SamPredictor(sam)

# Also prepare automatic mask generator for comparison/fallback
sam_auto_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side=32,
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
    min_mask_region_area=100
)

print(f"SAM loaded successfully on {DEVICE}")

## 5. Generate Segmentation Masks with SAM

Run SAM on each cropped image using point prompts (center of crop) combined with bounding box prompts (from YOLO annotations). SAM produces multiple candidate masks per crop — we select the highest-scoring one.

In [ ]:
# Generate SAM masks for all crops
crops_path = Path(CROPS_DIR)
sam_output_path = Path(SAM_OUTPUT_DIR)

# Create output directories
(sam_output_path / 'images').mkdir(parents=True, exist_ok=True)
(sam_output_path / 'masks').mkdir(parents=True, exist_ok=True)

# Load crop annotations
with open(crops_path / 'annotations.json') as f:
    crop_annotations = json.load(f)

print(f"Processing {len(crop_annotations)} crops with SAM...")

sam_annotations = {}
success_count = 0
fail_count = 0
scores_list = []

for sample_name, ann in tqdm(crop_annotations.items(), desc="Generating SAM masks"):
    img_path = crops_path / 'images' / sample_name
    
    if not img_path.exists():
        fail_count += 1
        continue
    
    image = cv2.imread(str(img_path))
    if image is None:
        fail_count += 1
        continue
    
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image_rgb.shape[:2]
    
    try:
        # Set image for predictor
        sam_predictor.set_image(image_rgb)
        
        # Point prompt at center of crop
        center_x, center_y = w // 2, h // 2
        input_point = np.array([[center_x, center_y]])
        input_label = np.array([1])  # 1 = foreground
        
        # Combined point + box prompt if rel_box is available
        rel_box = ann.get('rel_box')
        if rel_box:
            input_box = np.array(rel_box)
            masks, scores, _ = sam_predictor.predict(
                point_coords=input_point,
                point_labels=input_label,
                box=input_box,
                multimask_output=True
            )
        else:
            masks, scores, _ = sam_predictor.predict(
                point_coords=input_point,
                point_labels=input_label,
                multimask_output=True
            )
        
        # Select best mask
        best_idx = np.argmax(scores)
        best_mask = masks[best_idx].astype(np.uint8)
        best_score = float(scores[best_idx])
        
        # Save mask
        mask_filename = sample_name.replace('.png', '_mask.png').replace('.jpg', '_mask.png')
        cv2.imwrite(str(sam_output_path / 'masks' / mask_filename), best_mask * 255)
        
        # Copy original image
        cv2.imwrite(str(sam_output_path / 'images' / sample_name), image)
        
        # Update annotation
        new_ann = ann.copy()
        new_ann['mask_file'] = mask_filename
        new_ann['sam_score'] = best_score
        new_ann['mask_method'] = 'sam_point_box_prompt'
        sam_annotations[sample_name] = new_ann
        
        scores_list.append(best_score)
        success_count += 1
        
    except Exception as e:
        print(f"  Error: {sample_name}: {e}")
        fail_count += 1

# Save SAM annotations
with open(sam_output_path / 'annotations.json', 'w') as f:
    json.dump(sam_annotations, f, indent=2)

print(f"\n{'='*60}")
print(f"SAM ANNOTATION COMPLETE")
print(f"{'='*60}")
print(f"Successfully processed: {success_count}")
print(f"Failed: {fail_count}")
print(f"Output: {sam_output_path}")
print(f"{'='*60}")

## 6. Post-Process and Filter SAM Masks

Quality check the generated masks. Filter out low-confidence masks, inspect statistics, and visualize examples with overlays to verify mask quality before training.

In [ ]:
# SAM score statistics
sam_path = Path(SAM_OUTPUT_DIR)
with open(sam_path / 'annotations.json') as f:
    sam_annotations = json.load(f)

scores = [ann.get('sam_score', 0) for ann in sam_annotations.values()]

print(f"SAM Score Statistics ({len(scores)} masks):")
print(f"  Mean:   {np.mean(scores):.4f}")
print(f"  Median: {np.median(scores):.4f}")
print(f"  Min:    {np.min(scores):.4f}")
print(f"  Max:    {np.max(scores):.4f}")
print(f"  Std:    {np.std(scores):.4f}")

# Score distribution
low_quality = sum(1 for s in scores if s < 0.7)
medium_quality = sum(1 for s in scores if 0.7 <= s < 0.9)
high_quality = sum(1 for s in scores if s >= 0.9)
print(f"\n  Low quality (<0.7):    {low_quality} ({low_quality/len(scores)*100:.1f}%)")
print(f"  Medium quality (0.7-0.9): {medium_quality} ({medium_quality/len(scores)*100:.1f}%)")
print(f"  High quality (≥0.9):   {high_quality} ({high_quality/len(scores)*100:.1f}%)")

# Histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(scores, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(0.7, color='red', linestyle='--', label='Quality threshold (0.7)')
axes[0].set_xlabel('SAM Score')
axes[0].set_ylabel('Count')
axes[0].set_title('SAM Score Distribution')
axes[0].legend()

# Per-class score distribution
class_scores = defaultdict(list)
for ann in sam_annotations.values():
    class_scores[ann['class_name']].append(ann.get('sam_score', 0))

axes[1].boxplot([class_scores[c] for c in ['fiber', 'film', 'fragment']],
               labels=['fiber', 'film', 'fragment'])
axes[1].set_ylabel('SAM Score')
axes[1].set_title('SAM Score by Class')

plt.tight_layout()
plt.show()

# Visualize 2 samples per class (Original | Mask | Overlay)
by_class = defaultdict(list)
for name, ann in sam_annotations.items():
    by_class[ann['class_name']].append(name)

fig, axes = plt.subplots(3, 6, figsize=(24, 12))
fig.suptitle('SAM Mask Quality Check (Original | Mask | Overlay)', fontsize=14)

for row, cls_name in enumerate(['fiber', 'film', 'fragment']):
    samples = random.sample(by_class[cls_name], min(2, len(by_class[cls_name])))
    for j, name in enumerate(samples):
        ann = sam_annotations[name]
        col_offset = j * 3
        
        img = cv2.imread(str(sam_path / 'images' / name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        mask_file = ann.get('mask_file', name.replace('.png', '_mask.png'))
        mask = cv2.imread(str(sam_path / 'masks' / mask_file), cv2.IMREAD_GRAYSCALE)
        mask_binary = (mask > 127).astype(np.uint8)
        
        overlay = img.copy()
        overlay[mask_binary == 1] = (overlay[mask_binary == 1] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
        
        axes[row, col_offset].imshow(img)
        axes[row, col_offset].set_title(f"{cls_name} — Original", fontsize=9)
        axes[row, col_offset].axis('off')
        
        axes[row, col_offset+1].imshow(mask_binary, cmap='gray')
        axes[row, col_offset+1].set_title(f"Mask (score: {ann.get('sam_score',0):.3f})", fontsize=9)
        axes[row, col_offset+1].axis('off')
        
        axes[row, col_offset+2].imshow(overlay)
        axes[row, col_offset+2].set_title("Overlay", fontsize=9)
        axes[row, col_offset+2].axis('off')

plt.tight_layout()
plt.show()

## 7. Convert SAM Outputs to COCO Format Annotations

Convert SAM masks and bounding boxes into COCO-style annotations with proper `image_id`, `category_id`, `segmentation` (RLE), `bbox`, and `area` fields. This is useful for compatibility with evaluation tools like `pycocotools`.

In [ ]:
# Convert to COCO format for evaluation tools compatibility
coco_file = convert_to_coco_format(
    crops_dir=str(SAM_OUTPUT_DIR),
    output_file=str(Path(SAM_OUTPUT_DIR) / 'coco_annotations.json')
)

# Verify COCO annotations
with open(coco_file) as f:
    coco_data = json.load(f)

print(f"\nCOCO Annotations Summary:")
print(f"  Images:      {len(coco_data['images'])}")
print(f"  Annotations: {len(coco_data['annotations'])}")
print(f"  Categories:  {[c['name'] for c in coco_data['categories']]}")

# Count per category
cat_counts = defaultdict(int)
for ann in coco_data['annotations']:
    cat_id = ann['category_id']
    cat_name = next(c['name'] for c in coco_data['categories'] if c['id'] == cat_id)
    cat_counts[cat_name] += 1
for cat, count in sorted(cat_counts.items()):
    print(f"    {cat}: {count}")

## 8. Build Custom Dataset Class for Mask R-CNN

The `CropDataset` class loads cropped images and their corresponding SAM masks. It looks for masks in the `masks/` directory (written by SAM) and falls back to ellipse masks if none are found. Data augmentations include random flips, rotations, brightness/contrast, and Gaussian noise.

In [ ]:
# Create dataset from SAM-annotated crops
train_transforms = get_transforms(train=True, img_size=CROP_SIZE)
val_transforms = get_transforms(train=False, img_size=CROP_SIZE)

train_dataset = CropDataset(
    crops_dir=str(SAM_OUTPUT_DIR),
    transforms=train_transforms
)

print(f"\nDataset size: {len(train_dataset)} samples")

# Inspect a sample
sample_img, sample_target = train_dataset[0]
print(f"\nSample:")
print(f"  Image shape:  {sample_img.shape}")
print(f"  Boxes:        {sample_target['boxes']}")
print(f"  Labels:       {sample_target['labels']} ({CLASS_NAMES[sample_target['labels'][0].item()]})")
print(f"  Mask shape:   {sample_target['masks'].shape}")
print(f"  Mask nonzero: {sample_target['masks'].sum().item()} pixels")

# Visualize a few augmented training samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Training Samples (Augmented)', fontsize=14)

for i in range(8):
    idx = random.randint(0, len(train_dataset) - 1)
    img, target = train_dataset[idx]
    row, col = i // 4, i % 4
    
    # Denormalize image for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_display = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    
    # Overlay mask
    mask = target['masks'][0].numpy()
    overlay = img_display.copy()
    overlay[mask == 1] = overlay[mask == 1] * 0.5 + np.array([0, 1, 0]) * 0.5
    
    axes[row, col].imshow(overlay)
    label = target['labels'][0].item()
    axes[row, col].set_title(f"{CLASS_NAMES[label]}", fontsize=10)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## 9. Define Mask R-CNN Model Architecture

Load a pretrained Mask R-CNN (ResNet50-FPN backbone) and replace the classification and mask prediction heads to match our 4-class setup (background + fiber + film + fragment).

In [ ]:
# Create Mask R-CNN with custom head
model = get_model(NUM_CLASSES, pretrained=True)
model.to(DEVICE)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Mask R-CNN (ResNet50-FPN)")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Num classes:          {NUM_CLASSES} ({CLASS_NAMES})")
print(f"  Device:               {DEVICE}")

## 10. Configure Training Hyperparameters and Data Loaders

Set up the AdamW optimizer with cosine annealing LR scheduler, create the DataLoader with custom collate function, and configure all training hyperparameters.

In [ ]:
# DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=MASKRCNN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

# Optimizer — AdamW with weight decay
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=MASKRCNN_LR, weight_decay=0.0005)

# LR Scheduler — Cosine annealing from LR down to LR*0.01
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MASKRCNN_EPOCHS, eta_min=MASKRCNN_LR * 0.01
)

print(f"Training Configuration:")
print(f"  Epochs:          {MASKRCNN_EPOCHS}")
print(f"  Batch size:      {MASKRCNN_BATCH_SIZE}")
print(f"  Learning rate:   {MASKRCNN_LR}")
print(f"  Optimizer:       AdamW (weight_decay=0.0005)")
print(f"  LR Scheduler:    CosineAnnealing (eta_min={MASKRCNN_LR * 0.01})")
print(f"  Dataset size:    {len(train_dataset)}")
print(f"  Batches/epoch:   {len(train_loader)}")
print(f"  Save directory:  {SAVE_DIR}")

## 11. Train Mask R-CNN Model

Full training loop with loss tracking, gradient clipping, and automatic checkpointing of the best model.

In [ ]:
# Training loop
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
best_loss = float('inf')
history = {'epoch': [], 'loss': [], 'lr': [],
           'loss_classifier': [], 'loss_box_reg': [], 
           'loss_mask': [], 'loss_objectness': [], 'loss_rpn_box_reg': []}

print(f"\n{'='*60}")
print(f"TRAINING MASK R-CNN — {MASKRCNN_EPOCHS} epochs")
print(f"{'='*60}\n")

for epoch in range(MASKRCNN_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_losses = defaultdict(float)
    batch_count = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MASKRCNN_EPOCHS}")
    for images, targets in pbar:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        
        # Skip batches with empty boxes
        if not all(len(t['boxes']) > 0 for t in targets):
            continue
        
        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        optimizer.step()
        
        # Track losses
        batch_loss = losses.item()
        epoch_loss += batch_loss
        batch_count += 1
        for k, v in loss_dict.items():
            epoch_losses[k] += v.item()
        
        pbar.set_postfix({'loss': f'{batch_loss:.4f}'})
    
    lr_scheduler.step()
    
    # Epoch summary
    avg_loss = epoch_loss / max(batch_count, 1)
    current_lr = optimizer.param_groups[0]['lr']
    
    history['epoch'].append(epoch + 1)
    history['loss'].append(avg_loss)
    history['lr'].append(current_lr)
    for k in ['loss_classifier', 'loss_box_reg', 'loss_mask', 'loss_objectness', 'loss_rpn_box_reg']:
        history[k].append(epoch_losses.get(k, 0) / max(batch_count, 1))
    
    print(f"Epoch {epoch+1}/{MASKRCNN_EPOCHS} — Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")
    
    # Save checkpoint
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
        'history': history
    }
    torch.save(checkpoint, str(SAVE_DIR / 'maskrcnn_crops_latest.pth'))
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(checkpoint, str(SAVE_DIR / 'maskrcnn_crops_best.pth'))
        print(f"  → New best model saved! (loss: {avg_loss:.4f})")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"Best loss: {best_loss:.4f}")
print(f"Best model: {SAVE_DIR / 'maskrcnn_crops_best.pth'}")
print(f"{'='*60}")

## 12. Evaluate Model Performance

Analyze training metrics — overall loss curve, per-component losses (classifier, box regression, mask, objectness, RPN box reg), and learning rate schedule.

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Mask R-CNN Training Curves', fontsize=14)

epochs = history['epoch']

# Total loss
axes[0, 0].plot(epochs, history['loss'], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Total Loss')
axes[0, 0].set_title('Total Training Loss')
axes[0, 0].grid(True, alpha=0.3)

# Component losses
for key, color, label in [
    ('loss_classifier', 'r', 'Classifier'),
    ('loss_box_reg', 'g', 'Box Regression'),
    ('loss_mask', 'b', 'Mask'),
    ('loss_objectness', 'm', 'Objectness'),
    ('loss_rpn_box_reg', 'c', 'RPN Box Reg'),
]:
    if key in history and history[key]:
        axes[0, 1].plot(epochs, history[key], color=color, label=label, linewidth=1.5)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Component Losses')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# Learning rate
axes[1, 0].plot(epochs, history['lr'], 'g-', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].set_title('Learning Rate Schedule (Cosine Annealing)')
axes[1, 0].grid(True, alpha=0.3)

# Mask loss specifically (most important for segmentation quality)
if 'loss_mask' in history and history['loss_mask']:
    axes[1, 1].plot(epochs, history['loss_mask'], 'b-', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Mask Loss')
    axes[1, 1].set_title('Mask Loss (Key Metric for Segmentation)')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
print(f"\nFinal Training Metrics (Epoch {epochs[-1]}):")
print(f"  Total Loss:     {history['loss'][-1]:.4f}")
print(f"  Mask Loss:      {history['loss_mask'][-1]:.4f}")
print(f"  Classifier:     {history['loss_classifier'][-1]:.4f}")
print(f"  Box Regression: {history['loss_box_reg'][-1]:.4f}")
print(f"\nBest Loss: {best_loss:.4f}")

## 13. Visualize Predictions with Bounding Boxes and Masks

Run inference on sample test images, apply score thresholds, and visualize predicted instance segmentation masks, bounding boxes, class labels, and confidence scores overlaid on the original images.

In [ ]:
# Load best model for inference
from torchvision import transforms as T

checkpoint = torch.load(str(SAVE_DIR / 'maskrcnn_crops_best.pth'), map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model from epoch {checkpoint['epoch']} (loss: {checkpoint['loss']:.4f})")

# Inference transform (no augmentation, just resize + normalize)
inference_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((CROP_SIZE, CROP_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Select random test samples
sam_path = Path(SAM_OUTPUT_DIR)
with open(sam_path / 'annotations.json') as f:
    test_anns = json.load(f)

test_samples = random.sample(list(test_anns.keys()), min(8, len(test_anns)))

# Color map for classes
class_colors = {
    1: (255, 50, 50),   # fiber → red
    2: (50, 255, 50),   # film → green
    3: (50, 50, 255),   # fragment → blue
}

fig, axes = plt.subplots(len(test_samples), 4, figsize=(20, 4 * len(test_samples)))
if len(test_samples) == 1:
    axes = axes.reshape(1, -1)

fig.suptitle('Mask R-CNN Predictions vs SAM Ground Truth', fontsize=14, y=1.01)
columns = ['Original', 'SAM GT Mask', 'Predicted Mask', 'Prediction Overlay']

for col, title in enumerate(columns):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

for i, name in enumerate(test_samples):
    ann = test_anns[name]
    
    # Load original image
    img = cv2.imread(str(sam_path / 'images' / name))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Column 0: Original
    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_ylabel(f"{ann['class_name']}", fontsize=10, rotation=0, labelpad=50)
    axes[i, 0].axis('off')
    
    # Column 1: SAM ground truth mask
    mask_file = ann.get('mask_file', name.replace('.png', '_mask.png'))
    mask_path = sam_path / 'masks' / mask_file
    if mask_path.exists():
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        gt_binary = (gt_mask > 127).astype(np.uint8)
        gt_overlay = img_rgb.copy()
        gt_overlay[gt_binary == 1] = (gt_overlay[gt_binary == 1] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
        axes[i, 1].imshow(gt_overlay)
    axes[i, 1].axis('off')
    
    # Run inference
    img_tensor = inference_transform(img_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        predictions = model(img_tensor)[0]
    
    # Get predictions above threshold
    score_thresh = 0.3
    keep = predictions['scores'] > score_thresh
    
    if keep.sum() > 0:
        pred_masks = predictions['masks'][keep]
        pred_labels = predictions['labels'][keep]
        pred_scores = predictions['scores'][keep]
        pred_boxes = predictions['boxes'][keep]
        
        # Best prediction
        best_idx = 0  # Highest score (already sorted)
        best_mask = pred_masks[best_idx, 0].cpu().numpy()
        best_label = pred_labels[best_idx].item()
        best_score = pred_scores[best_idx].item()
        best_box = pred_boxes[best_idx].cpu().numpy().astype(int)
        pred_class = CLASS_NAMES[best_label] if best_label < len(CLASS_NAMES) else '?'
        
        # Column 2: Predicted mask
        axes[i, 2].imshow(best_mask > 0.5, cmap='gray')
        axes[i, 2].set_title(f"Pred: {pred_class} ({best_score:.2f})", fontsize=9)
        axes[i, 2].axis('off')
        
        # Column 3: Overlay with bbox
        img_resized = cv2.resize(img_rgb, (CROP_SIZE, CROP_SIZE))
        pred_binary = (best_mask > 0.5).astype(np.uint8)
        color = class_colors.get(best_label, (255, 255, 0))
        color_norm = np.array(color) / 255.0
        
        overlay = img_resized.copy().astype(np.float32) / 255.0
        overlay[pred_binary == 1] = overlay[pred_binary == 1] * 0.5 + color_norm * 0.5
        
        # Draw bounding box
        x1, y1, x2, y2 = best_box
        overlay_uint8 = (overlay * 255).astype(np.uint8)
        cv2.rectangle(overlay_uint8, (x1, y1), (x2, y2), color, 2)
        cv2.putText(overlay_uint8, f"{pred_class} {best_score:.2f}", 
                   (x1, max(y1-5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
        
        axes[i, 3].imshow(overlay_uint8)
        axes[i, 3].axis('off')
    else:
        axes[i, 2].text(0.5, 0.5, 'No detection', ha='center', va='center', fontsize=12)
        axes[i, 2].axis('off')
        axes[i, 3].text(0.5, 0.5, 'No detection', ha='center', va='center', fontsize=12)
        axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

## 14. Copy Results Back to Google Drive

Copy trained models and generated data from local Colab storage back to Google Drive for persistence. **Colab local storage is wiped when the runtime disconnects.**

In [ ]:
# ============================================================================
# COPY RESULTS BACK TO GOOGLE DRIVE
# ============================================================================
import shutil

DRIVE_ROOT = Path('/content/drive/MyDrive/mp-detect')

# 1. Copy trained model checkpoints
drive_experiments = DRIVE_ROOT / 'experiments'
drive_experiments.mkdir(parents=True, exist_ok=True)

for model_file in ['maskrcnn_crops_best.pth', 'maskrcnn_crops_latest.pth']:
    src = SAVE_DIR / model_file
    dst = drive_experiments / model_file
    if src.exists():
        shutil.copy2(str(src), str(dst))
        print(f"✓ Copied {model_file} → Drive")
    else:
        print(f"✗ {model_file} not found locally")

# 2. Copy crops dataset (optional — uncomment if you want to keep crops on Drive)
# drive_crops = DRIVE_ROOT / 'data' / 'crops'
# if not drive_crops.exists() and CROPS_DIR.exists():
#     print("Copying crops to Drive...")
#     shutil.copytree(str(CROPS_DIR), str(drive_crops))
#     print(f"✓ Copied crops → {drive_crops}")

# 3. Copy SAM-annotated crops (optional — uncomment to keep on Drive)
# drive_sam = DRIVE_ROOT / 'data' / 'crops_sam'
# if not drive_sam.exists() and SAM_OUTPUT_DIR.exists():
#     print("Copying SAM crops to Drive...")
#     shutil.copytree(str(SAM_OUTPUT_DIR), str(drive_sam))
#     print(f"✓ Copied SAM crops → {drive_sam}")

print(f"\n{'='*60}")
print("RESULTS SAVED TO GOOGLE DRIVE")
print(f"{'='*60}")
print(f"  Best model:   {drive_experiments / 'maskrcnn_crops_best.pth'}")
print(f"  Latest model: {drive_experiments / 'maskrcnn_crops_latest.pth'}")
print(f"{'='*60}")

## Done!

**Trained models saved to Google Drive:**
- **Best**: `/content/drive/MyDrive/mp-detect/experiments/maskrcnn_crops_best.pth`
- **Latest**: `/content/drive/MyDrive/mp-detect/experiments/maskrcnn_crops_latest.pth`

**How this notebook works (local copy strategy):**
1. Data is copied from Google Drive → Colab local storage (`/content/mp_data/`)
2. All processing (cropping, SAM, training) runs on fast local SSD — no Drive disconnects
3. After training, results are copied back to Google Drive

**Use in the full pipeline:**
```bash
python src/pipeline_inference.py \
    --input dev-test/stitched/s1.png \
    --yolo experiments/augmented_microplastic_yolo/weights/best.pt \
    --effnet experiments/efficientnet_best.pth \
    --maskrcnn experiments/maskrcnn_crops_best.pth
```

**Pipeline flow:**
1. **YOLO** detects all microplastics (bounding boxes)
2. **Crops** extracted from each detection
3. **EfficientNet** classifies each crop: fiber / film / fragment
4. **Mask R-CNN** segments each crop: precise pixel mask
5. **Size calculation**: area, length, width, perimeter, circularity